# F1 Video Processing Pipeline

This notebook implements an advanced video processing pipeline for F1 race footage, including:
- Intelligent frame extraction
- Quality assessment
- Multi-format support
- Memory-efficient processing
- Real-time optimization

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import time
from datetime import datetime
import logging
from tqdm.notebook import tqdm
import json
from PIL import Image
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
from dataclasses import dataclass
from typing import List, Tuple, Optional

## 1. Configuration

In [ ]:
@dataclass
class ProcessingConfig:
    # Frame extraction settings
    target_fps: int = 30
    min_scene_change_threshold: float = 0.3
    
    # Quality thresholds
    min_brightness: float = 0.2
    max_brightness: float = 0.8
    min_contrast: float = 0.3
    max_blur: float = 100
    
    # Processing settings
    batch_size: int = 32
    num_threads: int = 4
    buffer_size: int = 1000
    
    # Output settings
    output_size: Tuple[int, int] = (1920, 1080)
    preserve_metadata: bool = True

config = ProcessingConfig()

## 2. Quality Assessment Functions

In [ ]:
def assess_frame_quality(frame: np.ndarray) -> Tuple[bool, dict]:
    """Assess frame quality using multiple metrics."""
    quality_metrics = {}
    
    # Brightness assessment
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray) / 255.0
    quality_metrics['brightness'] = brightness
    
    # Contrast assessment
    contrast = np.std(gray) / 255.0
    quality_metrics['contrast'] = contrast
    
    # Blur detection using Laplacian variance
    blur = cv2.Laplacian(gray, cv2.CV_64F).var()
    quality_metrics['blur'] = blur
    
    # Overall quality check
    is_good = (
        config.min_brightness <= brightness <= config.max_brightness and
        contrast >= config.min_contrast and
        blur >= config.max_blur
    )
    
    return is_good, quality_metrics

## 3. Scene Change Detection

In [ ]:
class SceneChangeDetector:
    def __init__(self, threshold: float = 0.3):
        self.threshold = threshold
        self.prev_frame = None
        
    def detect_change(self, frame: np.ndarray) -> bool:
        """Detect if frame represents a significant scene change."""
        if self.prev_frame is None:
            self.prev_frame = frame
            return True
            
        # Convert to grayscale
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        prev_gray = cv2.cvtColor(self.prev_frame, cv2.COLOR_BGR2GRAY)
        
        # Calculate frame difference
        frame_diff = cv2.absdiff(curr_gray, prev_gray)
        change_percent = np.mean(frame_diff) / 255.0
        
        # Update previous frame
        self.prev_frame = frame
        
        return change_percent > self.threshold

## 4. Frame Processing Pipeline

In [ ]:
class FrameProcessor:
    def __init__(self, config: ProcessingConfig):
        self.config = config
        self.scene_detector = SceneChangeDetector(config.min_scene_change_threshold)
        self.frame_buffer = queue.Queue(maxsize=config.buffer_size)
        self.processed_frames = queue.Queue()
        self.metadata = {}
        
    def process_video(self, video_path: str, output_dir: str):
        """Process video with parallel frame extraction and quality assessment."""
        os.makedirs(output_dir, exist_ok=True)
        
        # Start worker threads
        with ThreadPoolExecutor(max_workers=self.config.num_threads) as executor:
            # Submit frame extraction task
            extract_future = executor.submit(self._extract_frames, video_path)
            
            # Submit frame processing tasks
            process_futures = []
            for _ in range(self.config.num_threads - 1):
                future = executor.submit(self._process_frame_batch, output_dir)
                process_futures.append(future)
            
            # Wait for completion
            extract_future.result()
            for future in process_futures:
                future.result()
        
        # Save metadata
        if self.config.preserve_metadata:
            self._save_metadata(output_dir)
    
    def _extract_frames(self, video_path: str):
        """Extract frames from video while maintaining memory efficiency."""
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_interval = max(1, int(fps / self.config.target_fps))
        
        with tqdm(total=total_frames, desc="Extracting frames") as pbar:
            frame_count = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                    
                if frame_count % frame_interval == 0:
                    # Resize frame if needed
                    if frame.shape[:2] != self.config.output_size:
                        frame = cv2.resize(frame, self.config.output_size)
                    
                    # Add to buffer
                    self.frame_buffer.put((frame_count, frame))
                
                frame_count += 1
                pbar.update(1)
        
        cap.release()
        # Signal completion
        for _ in range(self.config.num_threads - 1):
            self.frame_buffer.put(None)
    
    def _process_frame_batch(self, output_dir: str):
        """Process batches of frames with quality assessment."""
        batch = []
        
        while True:
            item = self.frame_buffer.get()
            if item is None:
                break
                
            frame_idx, frame = item
            
            # Assess quality and detect scene change
            is_good_quality, metrics = assess_frame_quality(frame)
            is_scene_change = self.scene_detector.detect_change(frame)
            
            if is_good_quality or is_scene_change:
                # Save frame
                frame_path = os.path.join(output_dir, f"frame_{frame_idx:06d}.jpg")
                cv2.imwrite(frame_path, frame)
                
                # Store metadata
                self.metadata[frame_idx] = {
                    'timestamp': frame_idx / self.config.target_fps,
                    'quality_metrics': metrics,
                    'is_scene_change': is_scene_change
                }
            
            self.frame_buffer.task_done()
    
    def _save_metadata(self, output_dir: str):
        """Save processing metadata to JSON file."""
        metadata_path = os.path.join(output_dir, 'metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump(self.metadata, f, indent=2)

## 5. Video Processing Example

In [ ]:
def process_f1_video(video_path: str, output_dir: str):
    """Process an F1 race video with the enhanced pipeline."""
    # Initialize processor
    processor = FrameProcessor(config)
    
    # Process video
    print(f"Processing video: {video_path}")
    start_time = time.time()
    
    processor.process_video(video_path, output_dir)
    
    processing_time = time.time() - start_time
    print(f"Processing completed in {processing_time:.2f} seconds")
    
    # Display sample frames and statistics
    display_processing_results(output_dir)

def display_processing_results(output_dir: str):
    """Display sample processed frames and statistics."""
    # Load metadata
    with open(os.path.join(output_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    
    # Plot quality metrics distribution
    metrics = {'brightness': [], 'contrast': [], 'blur': []}
    for frame_data in metadata.values():
        for metric, values in metrics.items():
            values.append(frame_data['quality_metrics'][metric])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (metric, values) in zip(axes, metrics.items()):
        ax.hist(values, bins=30)
        ax.set_title(f'{metric.capitalize()} Distribution')
    plt.tight_layout()
    plt.show()
    
    # Display sample frames
    frame_files = sorted(os.listdir(output_dir))[:5]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, frame_file in zip(axes, frame_files):
        if frame_file.endswith('.jpg'):
            img = cv2.imread(os.path.join(output_dir, frame_file))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.axis('off')
    plt.show()

## 6. Usage Example

In [ ]:
# Example usage
video_path = "path/to/f1_race.mp4"
output_dir = "extracted_frames"

# Process video
process_f1_video(video_path, output_dir)